# Metricas de rendimiento
---

- **Taller 2**: Regresion 'lineal'
- **Fecha de entrega**: 13 de Marzo de 2024.


In [ ]:
# El archivo de datos se consulta localmente; no requiere montar Google Drive.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

A continuación encontrará datos de crecimiento de E.coli (para más de crecimiento bacteriano, consulte: https://es.wikipedia.org/wiki/Crecimiento_bacteriano). Tendrá como variables independientes el tiempo en que fue tomada cada muestra y su volumen celular, por otra parte, el conteo celular para cada punto será la variable dependiente. Realice una regresión que le permita estimar la cantidad de células en cualquier momento.

In [ ]:
data = pd.read_csv('datos/E.coli growth data.csv')
data

#  1. Explore el conjunto de datos

Grafique la variable dependiente respecto a las independientes, ¿la tendencia es lineal?

pista: ambas respuestas son correctas



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Configurar el estilo de Seaborn
sns.set(style="darkgrid")

# Crear la figura y los ejes 3D
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# Crear el gráfico de dispersión 3D
scatter = ax.scatter(data['Time(h)'], data['Cell volume(nL)'], data['Cell counts'],
                     c=data['Cell counts'], cmap='viridis', s=50, alpha=0.7)

# Configurar etiquetas y título
ax.set_xlabel('Tiempo (h)', fontsize=12, labelpad=10)
ax.set_ylabel('Volumen Celular (nL)', fontsize=12, labelpad=10)
ax.set_zlabel('Conteo Celular', fontsize=12, labelpad=10)
plt.title('Relación entre el Tiempo, Volumen Celular y Conteo Celular', fontsize=14, pad=20)

# Añadir una barra de color
cbar = fig.colorbar(scatter, ax=ax, pad=0.1)
cbar.set_label('Conteo Celular', rotation=270, labelpad=15)

# Mejorar la perspectiva
ax.view_init(elev=20, azim=45)

# Ajustar el diseño
plt.tight_layout()

# Mostrar el gráfico
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Gráfico 2D: Relación entre Tiempo y Conteo Celular
plt.figure(figsize=(6, 4))
plt.scatter(data['Time(h)'], data['Cell counts'], color='b', label='Datos')
plt.xlabel('Time (h)')
plt.ylabel('Cell Counts')
plt.title('Relación entre el Tiempo y el Conteo Celular')
plt.legend()
plt.show()

# Gráfico 2D: Relación entre Volumen Celular y Conteo Celular
plt.figure(figsize=(6, 4))
plt.scatter(data['Cell volume(nL)'], data['Cell counts'], color='g', label='Datos')
plt.xlabel('Cell Volume (nL)')
plt.ylabel('Cell Counts')
plt.title('Relación entre el Volumen Celular y el Conteo Celular')
plt.legend()
plt.show()

# Gráfico 2D: Relación entre Tiempo y Volumen Celular
plt.figure(figsize=(6, 4))
plt.scatter(data['Time(h)'], data['Cell volume(nL)'], color='r', label='Datos')
plt.xlabel('Time (h)')
plt.ylabel('Cell Volume (nL)')
plt.title('Relación entre el Tiempo y el Volumen Celular')
plt.legend()
plt.show()


#  2. Separe su conjunto de datos

Realice un hold-out de 70%-30% (No use Sklearn).Grafique ambas conjuntos. ¿La tendencia es igual en ambos casos?

**Este punto solo será válido si se considera el balanceo durante el particionamiento.**


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def balanced_holdout(data, test_size=0.3):
    # 'Cell counts' es la variable objetivo para el balance
    sorted_data = data.sort_values('Cell counts')
    n = len(data)
    test_n = int(n * test_size)

    # Seleccionar cada k-ésimo elemento para el conjunto de prueba
    k = n // test_n
    test_indices = np.arange(k-1, n, k)

    test = data.iloc[test_indices]
    train = data.drop(test.index)

    return train, test


train, test = balanced_holdout(data)

def plot_3d_scatter(data, title):
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    scatter = ax.scatter(data['Time(h)'], data['Cell volume(nL)'], data['Cell counts'],
                         c=data['Cell counts'], cmap='viridis', s=50, alpha=0.7)

    ax.set_xlabel('Tiempo (h)', fontsize=12, labelpad=10)
    ax.set_ylabel('Volumen Celular (nL)', fontsize=12, labelpad=10)
    ax.set_zlabel('Conteo Celular', fontsize=12, labelpad=10)
    plt.title(title, fontsize=14, pad=20)

    cbar = fig.colorbar(scatter, ax=ax, pad=0.1)
    cbar.set_label('Conteo Celular', rotation=270, labelpad=15)

    ax.view_init(elev=20, azim=45)
    plt.tight_layout()

# Graficar conjunto de entrenamiento
plot_3d_scatter(train, 'Conjunto de Entrenamiento (70%)')
plt.show()

# Graficar conjunto de prueba
plot_3d_scatter(test, 'Conjunto de Prueba (30%)')
plt.show()

# Comparación de estadísticas básicas
print("Estadísticas del conjunto de entrenamiento:")
print(train.describe())
print("\nEstadísticas del conjunto de prueba:")
print(test.describe())

In [ ]:
def plot_2d_scatter(train, test):
    variables = ['Time(h)', 'Cell volume(nL)', 'Cell counts']
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('Comparación de Conjuntos de Entrenamiento y Prueba', fontsize=16)

    for i, (var1, var2) in enumerate([(variables[0], variables[1]),
                                      (variables[0], variables[2]),
                                      (variables[1], variables[2])]):
        # Gráfico para el conjunto de entrenamiento
        sns.scatterplot(data=train, x=var1, y=var2, ax=axes[0, i],
                        hue='Cell counts', palette='viridis', legend=False)
        axes[0, i].set_title(f'Entrenamiento: {var1} vs {var2}')

        # Gráfico para el conjunto de prueba
        sns.scatterplot(data=test, x=var1, y=var2, ax=axes[1, i],
                        hue='Cell counts', palette='viridis', legend=False)
        axes[1, i].set_title(f'Prueba: {var1} vs {var2}')

    # Añadir una barra de color común
    norm = plt.Normalize(data['Cell counts'].min(), data['Cell counts'].max())
    sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
    sm.set_array([])

    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label('Cell counts', rotation=270, labelpad=15)

    plt.tight_layout(rect=[0, 0.03, 0.9, 0.95])
    plt.show()

# Generar los gráficos 2D
plot_2d_scatter(train, test)

# 3. Proponga un modelo

Proponga un modelo para la estimación de crecimiento celular, Justifique su elección.

**recuerde los 5 criterios de Gauss-Markov, ¿su modelo lo cumple?**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm

# Asumo que 'data' es tu DataFrame con las columnas 'Time(h)', 'Cell volume(nL)', y 'Cell counts'

# Preparar los datos
X = data[['Time(h)', 'Cell volume(nL)']]
y = data['Cell counts']

# Función para detectar y eliminar outliers usando el método del IQR
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Aplicar eliminación de outliers a las columnas relevantes
# Create a copy of the DataFrame to avoid modifying the original
X_cleaned = X.copy()
X_cleaned = remove_outliers(X_cleaned, 'Time(h)')
X_cleaned = remove_outliers(X_cleaned, 'Cell volume(nL)')

# Remove outliers from y and align indices with X_cleaned
y_cleaned = remove_outliers(pd.DataFrame(y), 'Cell counts')
X_cleaned = X_cleaned.loc[y_cleaned.index]
y_cleaned = y_cleaned['Cell counts']

# Renombrar X_cleaned a X e y_cleaned a y
X = X_cleaned
y = y_cleaned

# Verificar el tamaño de los datos después de la limpieza
print(f"Datos originales: {data.shape}")
print(f"Datos después de la limpieza: {X.shape}")



# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:

# Crear y ajustar el modelo
model = LinearRegression()
model.fit(X_train, y_train)

# Realizar predicciones
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Evaluar el modelo
mse_train = mean_squared_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)

print("Resultados del modelo:")
print(f"MSE (entrenamiento): {mse_train:.2f}")
print(f"R2 (entrenamiento): {r2_train:.2f}")
print(f"MSE (prueba): {mse_test:.2f}")
print(f"R2 (prueba): {r2_test:.2f}")

# Análisis estadístico detallado usando statsmodels
X_sm = sm.add_constant(X)
model_sm = sm.OLS(y, X_sm).fit()
print(model_sm.summary())

# Visualización de residuos
residuals = y - model.predict(X)
plt.figure(figsize=(8, 4))
sns.scatterplot(x=model.predict(X), y=residuals)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Valores predichos')
plt.ylabel('Residuos')
plt.title('Gráfico de Residuos')
plt.show()

# Gráfico de valores reales vs predichos
plt.figure(figsize=(8, 4))
sns.scatterplot(x=y, y=model.predict(X))
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.xlabel('Valores reales')
plt.ylabel('Valores predichos')
plt.title('Valores Reales vs Predichos')
plt.show()

# Importancia de las características
importance = pd.DataFrame({'feature': X.columns, 'importance': model.coef_})
importance = importance.sort_values('importance', ascending=False)
plt.figure(figsize=(6, 3))
sns.barplot(x='importance', y='feature', data=importance)
plt.title('Importancia de las Características')
plt.show()

In [ ]:
# Transformar las variables independientes a escala logarítmica para el modelo Lin-log
X_train_lin_log = np.log1p(X_train)
X_test_lin_log = np.log1p(X_test)

# Crear y ajustar el modelo Lin-log
model_lin_log = LinearRegression()
model_lin_log.fit(X_train_lin_log, y_train)

# Realizar predicciones
y_pred_train_lin_log = model_lin_log.predict(X_train_lin_log)
y_pred_test_lin_log = model_lin_log.predict(X_test_lin_log)

# Evaluar el modelo Lin-log
mse_train_lin_log = mean_squared_error(y_train, y_pred_train_lin_log)
r2_train_lin_log = r2_score(y_train, y_pred_train_lin_log)
mse_test_lin_log = mean_squared_error(y_test, y_pred_test_lin_log)
r2_test_lin_log = r2_score(y_test, y_pred_test_lin_log)

print("Resultados del modelo Lin-log:")
print(f"MSE (entrenamiento): {mse_train_lin_log:.4f}")
print(f"R2 (entrenamiento): {r2_train_lin_log:.4f}")
print(f"MSE (prueba): {mse_test_lin_log:.4f}")
print(f"R2 (prueba): {r2_test_lin_log:.4f}")

# Análisis estadístico detallado usando statsmodels para Lin-log
X_sm_lin_log = sm.add_constant(X_train_lin_log)
model_sm_lin_log = sm.OLS(y_train, X_sm_lin_log).fit()
print(model_sm_lin_log.summary())


In [ ]:
# Calcular residuos para el modelo Lin-log
residuals_lin_log = y_train - y_pred_train_lin_log

plt.figure(figsize=(10, 5))
sns.scatterplot(x=y_pred_train_lin_log, y=residuals_lin_log)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Valores predichos (Lin-log)')
plt.ylabel('Residuos')
plt.title('Gráfico de Residuos (Modelo Lin-log)')
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(x=y_test, y=y_pred_test_lin_log)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Valores reales')
plt.ylabel('Valores predichos (Lin-log)')
plt.title('Valores Reales vs Predichos (Modelo Lin-log)')
plt.show()


In [ ]:
# Transformar tanto las variables independientes como dependientes a escala logarítmica para el modelo Log-log
X_train_log_log = np.log1p(X_train)
X_test_log_log = np.log1p(X_test)
y_train_log_log = np.log1p(y_train)
y_test_log_log = np.log1p(y_test)

# Crear y ajustar el modelo Log-log
model_log_log = LinearRegression()
model_log_log.fit(X_train_log_log, y_train_log_log)

# Realizar predicciones
y_pred_train_log_log = model_log_log.predict(X_train_log_log)
y_pred_test_log_log = model_log_log.predict(X_test_log_log)

# Evaluar el modelo Log-log
mse_train_log_log = mean_squared_error(y_train_log_log, y_pred_train_log_log)
r2_train_log_log = r2_score(y_train_log_log, y_pred_train_log_log)
mse_test_log_log = mean_squared_error(y_test_log_log, y_pred_test_log_log)
r2_test_log_log = r2_score(y_test_log_log, y_pred_test_log_log)

print("Resultados del modelo Log-log:")
print(f"MSE (entrenamiento): {mse_train_log_log:.4f}")
print(f"R2 (entrenamiento): {r2_train_log_log:.4f}")
print(f"MSE (prueba): {mse_test_log_log:.4f}")
print(f"R2 (prueba): {r2_test_log_log:.4f}")

# Análisis estadístico detallado usando statsmodels para Log-log
X_sm_log_log = sm.add_constant(X_train_log_log)
model_sm_log_log = sm.OLS(y_train_log_log, X_sm_log_log).fit()
print(model_sm_log_log.summary())


In [ ]:
# Convertir predicciones a escala original para el gráfico
y_pred_test_log_log_original = np.expm1(y_pred_test_log_log)
y_test_original = np.expm1(y_test_log_log)

plt.figure(figsize=(10, 5))
sns.scatterplot(x=y_test_original, y=y_pred_test_log_log_original)
plt.plot([y_test_original.min(), y_test_original.max()],
         [y_test_original.min(), y_test_original.max()], 'r--', lw=2)
plt.xlabel('Valores reales')
plt.ylabel('Valores predichos (Log-log)')
plt.title('Valores Reales vs Predichos (Modelo Log-log)')
plt.show()


In [ ]:
!pip install lmfit

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lmfit import Model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Suponiendo que 'data' es tu DataFrame con las columnas 'Time(h)', 'Cell volume(nL)', y 'Cell counts'
t = data['Time(h)'].values
volume = data['Cell volume(nL)'].values
y = data['Cell counts'].values

def gompertz_growth(t, K, r, t0):
    return K * np.exp(-np.exp(-r * (t - t0)))

# Crear un objeto Model
gmodel = Model(gompertz_growth)

# Proporcionar conjeturas iniciales y establecer límites min/max
params = gmodel.make_params(K=np.max(y), r=1, t0=np.median(t))
params['K'].min = 0
params['r'].min = 0
params['t0'].min = 0

# Ajustar el modelo a los datos
result = gmodel.fit(y, params, t=t)

# Extraer los parámetros ajustados
popt_gomp = [result.params['K'].value, result.params['r'].value, result.params['t0'].value]

# Predicciones
y_pred_gomp = gompertz_growth(t, *popt_gomp)

# Visualización
plt.figure(figsize=(8, 4))
plt.scatter(t, y, label='Datos observados')
plt.plot(t, y_pred_gomp, 'g-', label='Modelo de Gompertz')
plt.xlabel('Tiempo (h)')
plt.ylabel('Conteo Celular')
plt.title('Modelo de Crecimiento de Gompertz')
plt.legend()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit # import the missing module


In [ ]:
def verhulst_growth(t, volume, K, r, t0, a, b):
    return K / ((1 + a * np.exp(-r * (t - t0) * volume)) ** b)

# Ajuste del modelo de Verhulst
x_data = np.column_stack((train['Time(h)'], train['Cell volume(nL)']))
y_data = train['Cell counts'].values

popt_verh, _ = curve_fit(lambda x, K, r, t0, a, b: verhulst_growth(x[:, 0], x[:, 1], K, r, t0, a, b),
                        x_data, y_data,
                        p0=[np.max(y_data), 1, np.median(train['Time(h)']), 1, 1],
                        maxfev=10000)

# Predicciones en el conjunto de prueba
x_test_data = np.column_stack((test['Time(h)'], test['Cell volume(nL)']))
y_test_data = test['Cell counts'].values

y_pred_verh = verhulst_growth(x_test_data[:, 0], x_test_data[:, 1], *popt_verh)
# Visualización
plt.figure(figsize=(8, 4))
plt.scatter(test['Time(h)'], y_test_data, label='Datos observados')
plt.plot(test['Time(h)'], y_pred_verh, 'r', label='Modelo de Verhulst')
plt.xlabel('Tiempo (h)')
plt.ylabel('Conteo Celular')
plt.title('Modelo de Crecimiento de Verhulst con Dos Variables Independientes')
plt.legend()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Función de modelo exponencial múltiple
def exponential_model(X, a, b, c):
    time, volume = X
    return a * np.exp(b * time + c * volume)

# Extraer los datos del conjunto de entrenamiento
x_train_time = train['Time(h)'].values
x_train_volume = train['Cell volume(nL)'].values
y_train_counts = train['Cell counts'].values

# Ajustar el modelo exponencial múltiple
popt, pcov = curve_fit(exponential_model, (x_train_time, x_train_volume), y_train_counts, p0=(1, 0.01, 0.01))

# Obtener los parámetros ajustados
a, b, c = popt
print("Parámetros del modelo exponencial múltiple: a =", a, ", b =", b, ", c =", c)

# Predicciones usando el modelo ajustado
x_test_time = test['Time(h)'].values
x_test_volume = test['Cell volume(nL)'].values
y_pred_counts = exponential_model((x_test_time, x_test_volume), a, b, c)

# Graficar el ajuste del modelo
plt.figure(figsize=(10, 6))
plt.scatter(train['Time(h)'], train['Cell counts'], label='Datos de Entrenamiento', color='blue', alpha=0.6)
plt.scatter(test['Time(h)'], test['Cell counts'], label='Datos de Prueba', color='orange', alpha=0.6)
plt.scatter(test['Time(h)'], y_pred_counts, label='Predicciones del Modelo', color='red', alpha=0.6)
plt.xlabel('Tiempo (h)')
plt.ylabel('Conteo Celular')
plt.title('Regresión Exponencial Múltiple para Conteo Celular')
plt.legend()
plt.show()

# Calcular el error cuadrático medio (MSE) para evaluar el modelo
mse = np.mean((y_pred_counts - test['Cell counts'].values) ** 2)
print("Error Cuadrático Medio (MSE) del modelo: ", mse)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Definir el modelo logístico
def logistic_model(X, L, a, b, c):
    time, volume = X
    return L / (1 + np.exp(-(a + b * time + c * volume)))

# Ajuste del modelo logístico
popt, pcov = curve_fit(logistic_model, (x_train_time, x_train_volume), y_train_counts, p0=(1, 1, 0.01, 0.01))

# Obtener los parámetros ajustados
L, a, b, c = popt
print("Parámetros del modelo logístico: L =", L, ", a =", a, ", b =", b, ", c =", c)

# Predicciones usando el modelo ajustado
y_pred_logistic = logistic_model((x_test_time, x_test_volume), L, a, b, c)

# Graficar el ajuste del modelo logístico
plt.figure(figsize=(10, 6))
plt.scatter(train['Time(h)'], train['Cell counts'], label='Datos de Entrenamiento', color='blue', alpha=0.6)
plt.scatter(test['Time(h)'], test['Cell counts'], label='Datos de Prueba', color='orange', alpha=0.6)
plt.scatter(test['Time(h)'], y_pred_logistic, label='Predicciones del Modelo Logístico', color='red', alpha=0.6)
plt.xlabel('Tiempo (h)')
plt.ylabel('Conteo Celular')
plt.title('Regresión Logística Múltiple para Conteo Celular')
plt.legend()
plt.show()


In [ ]:
# Crear variables polinomiales
x_train_time_poly = np.vstack((x_train_time, x_train_time**2, x_train_time**3)).T
x_train_volume_poly = np.vstack((x_train_volume, x_train_volume**2, x_train_volume**3)).T
X_train_poly = np.column_stack((x_train_time_poly, x_train_volume_poly))

# Definir el modelo polinomial
def polynomial_model(X, a, b, c, d, e, f):
    time, time2, time3, volume, volume2, volume3 = X.T
    return a + b * time + c * time2 + d * time3 + e * volume + f * volume2

# Ajustar el modelo polinomial
popt, pcov = curve_fit(polynomial_model, X_train_poly, y_train_counts, p0=(1, 1, 1, 1, 1, 1))

# Obtener los parámetros ajustados
a, b, c, d, e, f = popt
print("Parámetros del modelo polinomial múltiple:", a, b, c, d, e, f)

# Crear variables polinomiales de prueba
x_test_time_poly = np.vstack((x_test_time, x_test_time**2, x_test_time**3)).T
x_test_volume_poly = np.vstack((x_test_volume, x_test_volume**2, x_test_volume**3)).T
X_test_poly = np.column_stack((x_test_time_poly, x_test_volume_poly))

# Predicciones usando el modelo ajustado
y_pred_poly = polynomial_model(X_test_poly, a, b, c, d, e, f)

# Graficar el ajuste del modelo polinomial
plt.figure(figsize=(10, 6))
plt.scatter(train['Time(h)'], train['Cell counts'], label='Datos de Entrenamiento', color='blue', alpha=0.6)
plt.scatter(test['Time(h)'], test['Cell counts'], label='Datos de Prueba', color='orange', alpha=0.6)
plt.scatter(test['Time(h)'], y_pred_poly, label='Predicciones del Modelo Polinomial', color='red', alpha=0.6)
plt.xlabel('Tiempo (h)')
plt.ylabel('Conteo Celular')
plt.title('Regresión Polinomial Múltiple para Conteo Celular')
plt.legend()
plt.show()



In [ ]:
# Definir una función de base radial (RBF)
def rbf_kernel(X, centers, sigma):
    # Calcular la distancia euclidiana entre X y los centros
    dist = np.linalg.norm(X[:, None] - centers, axis=2)
    return np.exp(-dist**2 / (2 * sigma**2))

# Crear centros para RBF
centers = np.column_stack((x_train_time, x_train_volume))

# Calcular la matriz de características RBF para el conjunto de entrenamiento
sigma = 1.0  # ancho de la RBF
Phi_train = rbf_kernel(np.column_stack((x_train_time, x_train_volume)), centers, sigma)

# Resolver el sistema lineal para encontrar los coeficientes
weights = np.linalg.lstsq(Phi_train, y_train_counts, rcond=None)[0]

# Calcular la matriz de características RBF para el conjunto de prueba
Phi_test = rbf_kernel(np.column_stack((x_test_time, x_test_volume)), centers, sigma)

# Predicciones usando RBF
y_pred_rbf = Phi_test @ weights

# Graficar el ajuste del modelo RBF
plt.figure(figsize=(10, 6))
plt.scatter(train['Time(h)'], train['Cell counts'], label='Datos de Entrenamiento', color='blue', alpha=0.6)
plt.scatter(test['Time(h)'], test['Cell counts'], label='Datos de Prueba', color='orange', alpha=0.6)
plt.scatter(test['Time(h)'], y_pred_rbf, label='Predicciones del Modelo RBF', color='red', alpha=0.6)
plt.xlabel('Tiempo (h)')
plt.ylabel('Conteo Celular')
plt.title('Regresión con Funciones de Base Radial (RBF) para Conteo Celular')
plt.legend()
plt.show()


In [ ]:

import pandas as pd

def check_gauss_markov(model_sm):
  """
  Verifica si un modelo cumple con los 5 criterios de Gauss-Markov.

  Args:
    model_sm: Un objeto statsmodels.api.OLS.

  Returns:
    Un DataFrame con los criterios y si se cumplen o no.
  """

  # Crear un diccionario para almacenar los resultados
  results = {
      'Criterio': [
          'Linealidad',
          'Exogeneidad',
          'No multicolinealidad',
          'Homocedasticidad',
          'Independencia de los errores'
      ],
      'Cumple': [False] * 5
  }

  # 1. Linealidad: Se asume que el modelo es lineal en los parámetros.
  results['Cumple'][0] = True  # Se asume que se cumple

  # 2. Exogeneidad: Las variables explicativas no están correlacionadas con el término de error.
  # Se puede verificar con la significancia de los coeficientes y la distribución de los residuos.
  # Se asume que se cumple en este caso.
  results['Cumple'][1] = True

  # 3. No multicolinealidad: Las variables explicativas no están altamente correlacionadas entre sí.
  # Se puede verificar con el factor de inflación de la varianza (VIF).
  if model_sm.condition_number < 100:  # Un valor bajo de condition_number indica baja multicolinealidad
      results['Cumple'][2] = True

  # 4. Homocedasticidad: La varianza del término de error es constante.
  # Se puede verificar con el gráfico de residuos vs valores predichos.
  # Se asume que se cumple en este caso.
  results['Cumple'][3] = True

  # 5. Independencia de los errores: Los errores no están correlacionados entre sí.
  # Se puede verificar con el test de Durbin-Watson.
  if model_sm.df_resid > 0 and model_sm.resid.shape[0] > 1:
      if 1.5 < model_sm.resid.shape[0] * (1 - model_sm.resid.shape[0] * model_sm.mse_resid / model_sm.mse_model) < 2.5:  # Durbin-Watson test
          results['Cumple'][4] = True

  # Convertir el diccionario a un DataFrame
  df_results = pd.DataFrame(results)
  return df_results

# Obtener los resultados de la verificación para el modelo lineal
df_linear_results = check_gauss_markov(model_sm)
print("Resultados del modelo lineal:")
print(df_linear_results)

# Obtener los resultados de la verificación para el modelo Lin-log
df_lin_log_results = check_gauss_markov(model_sm_lin_log)
print("\nResultados del modelo Lin-log:")
print(df_lin_log_results)

# Obtener los resultados de la verificación para el modelo Log-log
df_log_log_results = check_gauss_markov(model_sm_log_log)
print("\nResultados del modelo Log-log:")
print(df_log_log_results)


# 4. Evalue su modelo

Seleccione una métrica de evaluación, justifique su respuesta. ¿Esta métrica demuestra que su modelo obtiene un buen ajuste?

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Calcular MSE y MAE para el modelo lineal
mse_test = mean_squared_error(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)

print("Resultados del modelo lineal:")
print(f"MSE (prueba): {mse_test:.2f}")
print(f"MAE (prueba): {mae_test:.2f}")
print(f"R2 (prueba): {r2_test:.2f}")

# Calcular MSE y MAE para el modelo Lin-log
mse_test_lin_log = mean_squared_error(y_test, y_pred_test_lin_log)
mae_test_lin_log = mean_absolute_error(y_test, y_pred_test_lin_log)
r2_test_lin_log = r2_score(y_test, y_pred_test_lin_log)

print("\nResultados del modelo Lin-log:")
print(f"MSE (prueba): {mse_test_lin_log:.4f}")
print(f"MAE (prueba): {mae_test_lin_log:.4f}")
print(f"R2 (prueba): {r2_test_lin_log:.4f}")

# Calcular MSE y MAE para el modelo Log-log
mse_test_log_log = mean_squared_error(y_test_log_log, y_pred_test_log_log)
mae_test_log_log = mean_absolute_error(y_test_log_log, y_pred_test_log_log)
r2_test_log_log = r2_score(y_test_log_log, y_pred_test_log_log)

print("\nResultados del modelo Log-log:")
print(f"MSE (prueba): {mse_test_log_log:.4f}")
print(f"MAE (prueba): {mae_test_log_log:.4f}")
print(f"R2 (prueba): {r2_test_log_log:.4f}")


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Calcular el Error Cuadrático Medio (MSE)
mse_logistic = mean_squared_error(test['Cell counts'], y_pred_logistic)
print("Error Cuadrático Medio (MSE) del modelo logístico:", mse_logistic)

# Calcular el Error Absoluto Medio (MAE)
mae_logistic = mean_absolute_error(test['Cell counts'], y_pred_logistic)
print("Error Absoluto Medio (MAE) del modelo logístico:", mae_logistic)

# Calcular el Coeficiente de Determinación (R²)
r2_logistic = r2_score(test['Cell counts'], y_pred_logistic)
print("Coeficiente de Determinación (R²) del modelo logístico:", r2_logistic)


In [ ]:
# Calcular el Error Cuadrático Medio (MSE)
mse_poly = mean_squared_error(test['Cell counts'], y_pred_poly)
print("Error Cuadrático Medio (MSE) del modelo polinomial:", mse_poly)

# Calcular el Error Absoluto Medio (MAE)
mae_poly = mean_absolute_error(test['Cell counts'], y_pred_poly)
print("Error Absoluto Medio (MAE) del modelo polinomial:", mae_poly)

# Calcular el Coeficiente de Determinación (R²)
r2_poly = r2_score(test['Cell counts'], y_pred_poly)
print("Coeficiente de Determinación (R²) del modelo polinomial:", r2_poly)


In [ ]:
# Calcular el Error Cuadrático Medio (MSE)
mse_rbf = mean_squared_error(test['Cell counts'], y_pred_rbf)
print("Error Cuadrático Medio (MSE) del modelo RBF:", mse_rbf)

# Calcular el Error Absoluto Medio (MAE)
mae_rbf = mean_absolute_error(test['Cell counts'], y_pred_rbf)
print("Error Absoluto Medio (MAE) del modelo RBF:", mae_rbf)

# Calcular el Coeficiente de Determinación (R²)
r2_rbf = r2_score(test['Cell counts'], y_pred_rbf)
print("Coeficiente de Determinación (R²) del modelo RBF:", r2_rbf)


In [ ]:
# Calcular las métricas
mse_verh = mean_squared_error(y_test_data, y_pred_verh)
mae_verh = mean_absolute_error(y_test_data, y_pred_verh)
r2_verh = r2_score(y_test_data, y_pred_verh)

print("Error Cuadrático Medio (MSE) del modelo de Verhulst:", mse_verh)
print("Error Absoluto Medio (MAE) del modelo de Verhulst:", mae_verh)
print("Coeficiente de Determinación (R²) del modelo de Verhulst:", r2_verh)


In [ ]:
# Calcular las métricas
mse_gomp = mean_squared_error(y, y_pred_gomp)
mae_gomp = mean_absolute_error(y, y_pred_gomp)
r2_gomp = r2_score(y, y_pred_gomp)

print("Error Cuadrático Medio (MSE) del modelo de Gompertz:", mse_gomp)
print("Error Absoluto Medio (MAE) del modelo de Gompertz:", mae_gomp)
print("Coeficiente de Determinación (R²) del modelo de Gompertz:", r2_gomp)


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import median_absolute_error
import numpy as np
import pandas as pd

# Implementación manual de Huber Loss
def huber_loss(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    abs_error = np.abs(error)
    squared_loss = 0.5 * (error ** 2)
    linear_loss = delta * (abs_error - 0.5 * delta)
    return np.where(abs_error <= delta, squared_loss, linear_loss).mean()

# Implementación manual de Quantile Loss
def quantile_loss(y_true, y_pred, alpha=0.5):
    error = y_true - y_pred
    return np.mean(np.maximum(alpha * error, (alpha - 1) * error))

# Implementación manual de Log-Cosh Loss
def log_cosh_loss(y_true, y_pred):
    error = y_true - y_pred
    return np.mean(np.log(np.cosh(error)))

def calculate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    adjusted_r2 = np.nan  # Se establece como NaN por ahora
    median_ae = median_absolute_error(y_true, y_pred)
    huber = huber_loss(y_true, y_pred)
    quantile = quantile_loss(y_true, y_pred, alpha=0.5)
    log_cosh = log_cosh_loss(y_true, y_pred)

    metrics = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'R²': r2,
        'Median Absolute Error': median_ae,
        'Huber Loss': huber,
        'Quantile Loss': quantile,
        'Log-Cosh Loss': log_cosh
    }
    return metrics

# Calcular las métricas para cada modelo (reemplaza estos valores con los datos reales)
linear_metrics = calculate_metrics(y_test, y_pred_test)
lin_log_metrics = calculate_metrics(y_test, y_pred_test_lin_log)
log_log_metrics = calculate_metrics(y_test_log_log, y_pred_test_log_log)
logistic_metrics = calculate_metrics(test['Cell counts'], y_pred_logistic)
poly_metrics = calculate_metrics(test['Cell counts'], y_pred_poly)
rbf_metrics = calculate_metrics(test['Cell counts'], y_pred_rbf)
verhulst_metrics = calculate_metrics(y_test_data, y_pred_verh)
gompertz_metrics = calculate_metrics(y, y_pred_gomp)

# Crear un DataFrame con las métricas
data = {
    'Modelo': ['Lineal', 'Lin-log', 'Log-log', 'Logístico', 'Polinomial', 'RBF', 'Verhulst', 'Gompertz'],
    'MSE': [linear_metrics['MSE'], lin_log_metrics['MSE'], log_log_metrics['MSE'], logistic_metrics['MSE'], poly_metrics['MSE'], rbf_metrics['MSE'], verhulst_metrics['MSE'], gompertz_metrics['MSE']],
    'RMSE': [linear_metrics['RMSE'], lin_log_metrics['RMSE'], log_log_metrics['RMSE'], logistic_metrics['RMSE'], poly_metrics['RMSE'], rbf_metrics['RMSE'], verhulst_metrics['RMSE'], gompertz_metrics['RMSE']],
    'MAE': [linear_metrics['MAE'], lin_log_metrics['MAE'], log_log_metrics['MAE'], logistic_metrics['MAE'], poly_metrics['MAE'], rbf_metrics['MAE'], verhulst_metrics['MAE'], gompertz_metrics['MAE']],
    'MAPE': [linear_metrics['MAPE'], lin_log_metrics['MAPE'], log_log_metrics['MAPE'], logistic_metrics['MAPE'], poly_metrics['MAPE'], rbf_metrics['MAPE'], verhulst_metrics['MAPE'], gompertz_metrics['MAPE']],
    'R²': [linear_metrics['R²'], lin_log_metrics['R²'], log_log_metrics['R²'], logistic_metrics['R²'], poly_metrics['R²'], rbf_metrics['R²'], verhulst_metrics['R²'], gompertz_metrics['R²']],
    'Median Absolute Error': [linear_metrics['Median Absolute Error'], lin_log_metrics['Median Absolute Error'], log_log_metrics['Median Absolute Error'], logistic_metrics['Median Absolute Error'], poly_metrics['Median Absolute Error'], rbf_metrics['Median Absolute Error'], verhulst_metrics['Median Absolute Error'], gompertz_metrics['Median Absolute Error']],
    'Huber Loss': [linear_metrics['Huber Loss'], lin_log_metrics['Huber Loss'], log_log_metrics['Huber Loss'], logistic_metrics['Huber Loss'], poly_metrics['Huber Loss'], rbf_metrics['Huber Loss'], verhulst_metrics['Huber Loss'], gompertz_metrics['Huber Loss']],
    'Quantile Loss': [linear_metrics['Quantile Loss'], lin_log_metrics['Quantile Loss'], log_log_metrics['Quantile Loss'], logistic_metrics['Quantile Loss'], poly_metrics['Quantile Loss'], rbf_metrics['Quantile Loss'], verhulst_metrics['Quantile Loss'], gompertz_metrics['Quantile Loss']],
    'Log-Cosh Loss': [linear_metrics['Log-Cosh Loss'], lin_log_metrics['Log-Cosh Loss'], log_log_metrics['Log-Cosh Loss'], logistic_metrics['Log-Cosh Loss'], poly_metrics['Log-Cosh Loss'], rbf_metrics['Log-Cosh Loss'], verhulst_metrics['Log-Cosh Loss'], gompertz_metrics['Log-Cosh Loss']]
}

df_metrics = pd.DataFrame(data)
df_metrics



MSE: El modelo Log-log tiene el menor MSE, indicando que tiene el menor error cuadrático medio.
RMSE: El modelo Log-log también tiene el menor RMSE, sugiriendo un buen ajuste en términos de error cuadrático medio.
MAE: El modelo Log-log tiene el menor MAE, mostrando el menor error absoluto promedio.
R²: El modelo Polinomial y RBF tienen el mayor R², indicando que explican mejor la varianza de los datos.
Median Absolute Error: El modelo Log-log tiene el menor Median Absolute Error, lo que es favorable.